# H1: styl życia a wiek i zainteresowania w predykcji prawego swipe'a

## O co pytamy (po ludzku)

W aplikacji osoba **A** przegląda profil osoby **B** i przesuwa go w prawo (zainteresowana) albo w lewo (nie).
Hipoteza **H1** stawia pytanie: **czy zgodność stylu życia przewiduje decyzję o akceptacji (prawy swipe) silniej niż zgodność wieku i zainteresowań?**

Intuicja stojąca za hipotezą zakłada, że w perspektywie wspólnego mieszkania kluczowe są codzienne nawyki (np. palenie, pora snu, poziom hałasu, czystość), a nie to, czy ktoś byłby dobrym znajomym na gruncie hobby czy zbliżonego wieku.

Chcemy zweryfikować to empirycznie poprzez:
1. Porównanie wskaźników akceptacji w grupach zgodności stylu życia, wieku i zainteresowań z przedziałami ufności (bootstrap klastrowany po użytkownikach).
2. Oszacowanie względnej siły predykcyjnej bloków w regresji logistycznej z dwustronnie klastrowanymi błędami standardowymi (**D16, D18**).
3. Test z tasowaniem cech odbiorcy (shuffled null baseline) — test na przypadek.
4. Sprawdzenie uogólniania na nowych użytkowników w 5-krotnej walidacji krzyżowej (**D18**).

## Status: badanie hipotezy głównej (konfirmacyjne / eksploracyjne)

Kierunek relacji i definicje bloków zostały sformalizowane w Karcie Eksploracji M1 i Codebooku przed przeprowadzeniem modelowania. Zgodnie z zasadami kursu AMC, dobrze uzasadniony wynik negatywny (np. brak dominacji stylu życia nad demografią) stanowi pełnoprawny, wartościowy wynik naukowy.

## Jednostka i zmienna wynikowa

Jeden wiersz to jeden skierowany swipe A→B na profil użytkownika. Główną zmienną wynikową jest `y_right` (1 = prawy swipe, 0 = lewy swipe).

## Decyzje zespołu zastosowane tutaj

(`docs/prompts/m1-generation-plan.md` i `docs/codebook.md`):
- cechy osoby bierzemy z wersji profilu **z dnia swipe'a**, z flagą źródła (**D7**),
- konta z miastem Hyderabad wyrzucamy z próby głównej (**D10**),
- płeć `O` to osobna kategoria (**D8**), pasma wieku 27+ łączymy w jedno (**D9**),
- w analizie głównej **nie** wykluczamy kont skrajnych ani mało aktywnych (**D6**),
- bez progu „bliskie zeru” i bez modelu dwuetapowego (**D17**),
- **D12 i D13**: tagi stylu życia dublujące pola strukturalne uzupełniają braki w tych polach, a kolumna `interests` zostaje oczyszczona z tagów stylu życia i statusu studenta (brak redundancji!),
- **D16**: porównanie bloków za pomocą 1–3 skalarów na blok, testowane w **obu kolejnościach** włączania.

## Estymator (D18)

Regresja logistyczna z błędami standardowymi **klastrowanymi dwustronnie po nadawcy i odbiorcy** (`user_key` i `target_user_key`), ponieważ ta sama osoba występuje wielokrotnie w zbiorze. Dodatkowo estymacja na nowych użytkownikach przez `GroupKFold` po nadawcach.

## Prywatność i zakres zmiennych

Korzystamy **wyłącznie z kolumn niezbędnych dla H1**:
- styl życia: `lifestyle_agree_ratio`, `lifestyle_tag_conflict`, `lifestyle_shared_tags_n`,
- wiek: `same_band`, `age_band_diff`,
- zainteresowania: `interests_jaccard`, `interests_shared_n`, `interests_count_diff_abs`,
- kontrola bazowa: `same_sex`, `same_city`.
Żadne pola bio nie są wczytywane ani analizowane (zarezerwowane dla H3). Do raportu trafiają wyłącznie agregaty.


In [ ]:
from pathlib import Path

from rentfriends import h1

df = h1.load_h1_data(Path("../data/processed"))
print("Kolumny datasetu utworzonego pod H1:")
print(list(df.columns))

## Słownik zmiennych i sens istnienia kolumn w zbiorze danych H1

Poniższe zestawienie wyjaśnia znaczenie operacyjne oraz sens istnienia w badaniu każdej z kolumn analitycznych przygotowanych w zbiorze `h1_analysis_dataset.parquet` 

### 1. Rekonstrukcja profilu as-of w czasie i analiza wrażliwości (D7, D6)
- **`source_flag_receiver`** (kategorie: `prior_version`, `first_version`, `users_current`): Źródło cech profilu odbiorcy. Wskazuje, czy profil pochodzi ściśle z wersji sprzed dnia decyzji (`prior_version` — brak wycieku danych z przyszłości), z pierwszej zarejestrowanej wersji (`first_version`), czy z tabeli bieżącej (`users_current`). Umożliwia weryfikację odporności (**D7**).
- **`source_flag_sender`** (kategorie: `prior_version`, `first_version`, `users_current`): Analogiczna flaga źródła profilu dla nadawcy swipe'a (**D7**).
- **`both_prior_version`** (0/1): Binarna flaga oznaczająca, że cechy profilu **obu** stron pochodzą ściśle ze stanu sprzed dnia decyzji. Służy jako filtr w najbardziej restrykcyjnym scenariuszu analizy wrażliwości (**D7**).
- **`sender_total_swipes`** (int): Łączna liczba swipe'ów wykonanych przez danego nadawcę. Pozwala kontrolować intensywność korzystania z aplikacji.
- **`sender_right_rate`** (float $[0, 1]$): Odsetek prawych swipe'ów nadawcy. Mierzy indywidualny poziom selektywności użytkownika (od osób bardzo wybrednych po klikające bez selekcji).
- **`is_low_activity`** (0/1): Flaga użytkownika o małej aktywności ($< 10$ swipe'ów). Służy w analizie wrażliwości (**D6**) do weryfikacji, czy incydentalni użytkownicy nie zniekształcają wyników.
- **`is_extreme_user`** (0/1): Flaga użytkownika o skrajnym zachowaniu ($0\%$ lub $100\%$ prawych swipe'ów — potencjalne boty lub użytkownicy nieselektywni). Używana do testu odporności w analizie wrażliwości (**D6**).

### 2. Kontrola demograficzna: Płeć (D8)
- **`sex_sender`** (kategorie: `F`, `M`, `O`): Płeć nadawcy swipe'a (zgodnie z **D8** płeć `O` stanowi odrębną kategorię). Podstawowa zmienna demograficzna.
- **`sex_receiver`** (kategorie: `F`, `M`, `O`): Płeć odbiorcy swipe'a.
- **`same_sex`** (0/1): Flaga tej samej płci nadawcy i odbiorcy. Kluczowa zmienna zakłócająca — preferencje dotyczące płci współlokatora (np. kobiety poszukujące wyłącznie współlokatorek) są jednym z najsilniejszych filtrów wstępnych na rynku najmu.
- **`sex_pair`** (kategorie: np. `F-F`, `M-M`, `F-M`, `M-F` itp.): Kategoryczny typ relacji płciowej w parze. Pozwala na pełną kontrolę interakcji płciowych w eksploracji i tabelach krzyżowych.

### 3. Blok Wieku (konkurencyjny wymiar demograficzny, D9)
- **`age_band_sender`** (kategorie: `18-20`, `21-23`, `24-26`, `27+`): Pasmo wieku nadawcy (po złączeniu pasm $27+$ wg **D9**).
- **`age_band_receiver`** (kategorie): Pasmo wieku odbiorcy (po złączeniu pasm $27+$ wg **D9**).
- **`same_band`** (0/1): Flaga tożsamości pasma wieku obu użytkowników. Podstawowa miara homofilii wiekowej — testuje, czy użytkownicy preferują osoby na dokładnie tym samym etapie życia.
- **`age_band_diff`** (int): Kierunkowa różnica rang pasm wieku ($\text{odbiorca} - \text{nadawca}$). Mierzy kierunkową asymetrię wieku (preferencję dla osób starszych [wartości dodatnie] vs młodszych [wartości ujemne]).
- **`age_band_diff_abs`** (int): Bezwzględna różnica rang pasm wieku ($|\text{diff}|$). Mierzy odległość pokoleniową między użytkownikami niezależnie od kierunku.

### 4. Kontrola geograficzna: Miasto (D10)
- **`city_sender`** (str): Miasto zadeklarowane przez nadawcę (po wykluczeniu kont z Hyderabadu wg **D10**).
- **`city_receiver`** (str): Miasto zadeklarowane przez odbiorcę (po wykluczeniu kont z Hyderabadu wg **D10**).
- **`same_city`** (kategorie: `same`, `different`, `unknown`): Kategoryczna zgodność lokalizacji nadawcy i odbiorcy.
- **`city_same`** (0/1): Binarna zmienna dummy oznaczająca, że nadawca i odbiorca deklarują to samo znane miasto. Używana w regresji logistycznej do wychwycenia premii za wspólne miasto.
- **`city_diff`** (0/1): Binarna zmienna dummy oznaczająca, że nadawca i odbiorca deklarują różne znane miasta. Używana w regresji logistycznej do kontroli kosztu poszukiwania lokalu w innym mieście.

### 5. Blok Stylu Życia (główny badany wymiar hipotezy H1, D12, D13)
- **`lifestyle_agreed_count`** (int): Liczba cech stylu życia, w których obie strony mają identyczną deklarację (po uzupełnieniu braków tagami wg **D13**). Licznik zgodności nawyków (czystość, hałas, palenie, alkohol, rytm dnia, praca zdalna).
- **`lifestyle_known_count`** (int): Liczba cech stylu życia zadeklarowanych przez obie osoby. Stanowi mianownik wskaźnika zgodności i zabezpiecza przed zniekształceniem estymacji przez brak danych.
- **`lifestyle_agree_ratio`** (float $[0, 1]$): Syntetyczny wskaźnik zgodności codziennych nawyków ($\frac{\text{lifestyle\_agreed\_count}}{\text{lifestyle\_known\_count}}$). Główny skalar mierzący kompatybilność współlokatorską w hipotezie H1.
- **`lifestyle_shared_tags_n`** (int): Liczba wspólnych tagów tożsamościowych związanych ze stylem życia. Mierzy afirmatywne podobieństwo deklaracji.
- **`lifestyle_tag_conflict`** (0/1): Binarna flaga bezpośredniego konfliktu kierunkowego na kluczowych nawykach (np. palący vs niepalący, głośny vs cichy, ranny ptaszek vs nocny marek). W odróżnieniu od ogólnej zgodności wychwytuje tzw. *deal-breakery*, które mogą natychmiastowo dyskwalifikować drugą osobę.

### 6. Blok Zainteresowań i Hobby (konkurencyjny wymiar behawioralny, D12, D13)
- **`interests_jaccard`** (float $[0, 1]$): Indeks podobieństwa Jaccarda ($\frac{|A \cap B|}{|A \cup B|}$) liczony wyłącznie na czystych zainteresowaniach (po usunięciu tagów stylu życia wg **D12/D13**). Główny znormalizowany wskaźnik zbieżności pasji i tematów do rozmowy w konkurencyjnym bloku H1.
- **`interests_shared_n`** (int): Bezwzględna liczba wspólnych zainteresowań ($|A \cap B|$). Mierzy wolumen wspólnych pasji (pozwala odróżnić parę o 3 wspólnych hobby od pary o 1 wspólnym hobby przy identycznym indeksie Jaccarda).
- **`interests_count_sender`** (int): Liczba zainteresowań zadeklarowanych przez nadawcę. Wskaźnik zaangażowania profilowego i ekspresji nadawcy.
- **`interests_count_receiver`** (int): Liczba zainteresowań zadeklarowanych przez odbiorcę. Wskaźnik bogactwa profilu odbiorcy (widoczny na pierwszej karcie).
- **`interests_count_diff`** (int): Kierunkowa różnica liczby zainteresowań ($\text{odbiorca} - \text{nadawca}$). Mierzy asymetrię ekspresji hobby.
- **`interests_count_diff_abs`** (int): Bezwzględna różnica liczby zainteresowań. Kontroluje dysproporcję w nakładzie pracy i stopniu wypełnienia profilu między stronami.


In [ ]:
# %load_ext autoreload
# %autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rentfriends import h1

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

SEED = 42  # stałe ziarno losowości dla bootstrapu, CV i permutacji
N_BOOT = 1000
N_PERM = 500
N_FOLDS = 5

# Paleta kolorów ze STYLEGUIDE.md
PURPLE = "#A247E3"
PURPLE_INK = "#8B34C4"
BLUE = "#234CE0"
GRAY = "#595959"
HAIR = "#D5D2CB"
CORAL = "#D9534F"

DATA_DIR = Path("../data/processed")

## 1. Wczytanie danych

Wczytujemy przygotowany zbiór analityczny `h1_analysis_dataset.parquet`, zawierający cechy zrekonstruowane as-of (D7), z wykluczeniem Hyderabadu (D10) oraz z usuniętą redundancją tagów i pól strukturalnych (D12, D13).


In [ ]:
df = h1.load_h1_data(DATA_DIR)
print(f"Wczytano zbiór danych dla hipotezy H1: {df.shape[0]:,} obserwacji, {df.shape[1]} kolumn.")

# Weryfikacja: zbiór nie zawiera pól bio
assert not any("bio" in c for c in df.columns), "Błąd: wykryto niepożądane kolumny bio!"
print("Sprawdzenie poprawności zmiennych H1: brak kolumn bio, brak Hyderabadu.")

## 2. Tabela do analizy i filtry (Lejek próby)

**Co tu się dzieje:** Z surowych swipe'ów wybieramy wyłącznie swipe'y skierowane na profile (odrzucając swipe'y na ogłoszenia), usuwamy anomalne konta z Hyderabadu (**D10**) i każdemu swipe'owi przypisujemy cechy nadawcy oraz odbiorcy **z dnia swipe'a** (**D7**).

Poniższy lejek obrazuje selekcję próby analitycznej dla H1.


In [ ]:
swipes_raw = pd.read_parquet(DATA_DIR / "swipes.parquet")
users_raw = pd.read_parquet(DATA_DIR / "users.parquet")
hyd_users = set(users_raw.loc[users_raw["city"] == "Hyderabad", "user_key"])

profile_mask = (swipes_raw["target_type"] == "profile") & swipes_raw["target_user_key"].notna()
profile_swipes = swipes_raw[profile_mask]

funnel = pd.Series(
    {
        "1. Wszystkie swipe'y w bazie": len(swipes_raw),
        "2. Swipe'y skierowane na profile": len(profile_swipes),
        "3. Po wykluczeniu kont z Hyderabadu (D10)": len(df),
        "4. Kompletne obserwacje w modelu H1": int(
            df[["lifestyle_agree_ratio", "interests_jaccard", "same_band"]]
            .notna()
            .all(axis=1)
            .sum()
        ),
    }
)
display(funnel.to_frame("Liczba swipe'ów"))

print("\nRozkład źródeł wersji profilu as-of (D7) dla odbiorcy:")
print(df["source_flag_receiver"].value_counts(normalize=True).apply(lambda p: f"{p:.1%}"))

## 3. Eksploracja: wskaźniki akceptacji wg stylu życia, wieku i zainteresowań

**Jak czytać:** Badamy odsetek prawych swipe'ów w grupach wyznaczonych przez cechy trzech bloków:
1. **Styl życia**: konflikt kierunkowy (`lifestyle_tag_conflict`: 0 = brak konfliktu, 1 = konflikt) oraz odsetek zgodności pól (`lifestyle_agree_ratio`).
2. **Zainteresowania**: indeks podobieństwa Jaccarda (`interests_jaccard`) oraz liczba wspólnych hobby (`interests_shared_n`).
3. **Wiek**: zgodność pasma (`same_band`) oraz różnica pasm wieku (`age_band_diff`).

Przedziały ufności liczymy za pomocą **bootstrapu klastrowanego po nadawcach** (1000 powtórzeń), co zapobiega zaniżeniu błędu standardowego wynikającemu z wielokrotnych swipe'ów tej samej osoby.


In [ ]:
# 1. Styl życia: konflikt kierunkowy
tbl_conflict = h1.cluster_bootstrap_rate_table(
    df, "lifestyle_tag_conflict", n_boot=N_BOOT, seed=SEED
)
tbl_conflict["kategoria"] = tbl_conflict["kategoria"].map(
    {0: "Brak konfliktu (0)", 1: "Konflikt (1)"}
)

# 2. Styl życia: grupy zgodności pól (tercyle)
df["agree_tercyle"] = pd.qcut(
    df["lifestyle_agree_ratio"].fillna(0),
    q=3,
    labels=["Niski [0 - 0.33]", "Średni (0.33 - 0.50]", "Wysoki (0.50 - 1.00]"],
)
tbl_agree = h1.cluster_bootstrap_rate_table(df, "agree_tercyle", n_boot=N_BOOT, seed=SEED)

# 3. Zainteresowania: kwantyle Jaccarda
df["jaccard_group"] = pd.qcut(
    df["interests_jaccard"],
    q=[0, 0.75, 1.0],
    labels=["Niski / brak Jaccarda", "Wysoki Jaccard (> 0.125)"],
)
tbl_jaccard = h1.cluster_bootstrap_rate_table(df, "jaccard_group", n_boot=N_BOOT, seed=SEED)

# 4. Wiek: zgodność pasma
tbl_age = h1.cluster_bootstrap_rate_table(df, "same_band", n_boot=N_BOOT, seed=SEED)
tbl_age["kategoria"] = tbl_age["kategoria"].map({0.0: "Różne pasmo (0)", 1.0: "To samo pasmo (1)"})

print("--- Odsetek prawych swipe'ów wg konfliktu stylu życia ---")
display(tbl_conflict[["kategoria", "n_swipe", "odsetek_prawych", "ci_low_boot", "ci_high_boot"]])

print("\n--- Odsetek prawych swipe'ów wg zgodności pól stylu życia ---")
display(tbl_agree[["kategoria", "n_swipe", "odsetek_prawych", "ci_low_boot", "ci_high_boot"]])

print("\n--- Odsetek prawych swipe'ów wg podobieństwa zainteresowań ---")
display(tbl_jaccard[["kategoria", "n_swipe", "odsetek_prawych", "ci_low_boot", "ci_high_boot"]])

print("\n--- Odsetek prawych swipe'ów wg zgodności wieku ---")
display(tbl_age[["kategoria", "n_swipe", "odsetek_prawych", "ci_low_boot", "ci_high_boot"]])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8), sharey=True)

# Wykres 1: Styl życia (konflikt)
axes[0].bar(
    tbl_conflict["kategoria"], tbl_conflict["odsetek_prawych"], color=[BLUE, CORAL], width=0.5
)
yerr_c = [
    tbl_conflict["odsetek_prawych"] - tbl_conflict["ci_low_boot"],
    tbl_conflict["ci_high_boot"] - tbl_conflict["odsetek_prawych"],
]
axes[0].errorbar(
    tbl_conflict["kategoria"],
    tbl_conflict["odsetek_prawych"],
    yerr=yerr_c,
    fmt="none",
    color=GRAY,
    capsize=4,
    lw=1.5,
)
axes[0].set_title("Styl życia (konflikt)")
axes[0].set_ylabel("P(prawy swipe)")
axes[0].set_ylim(0, 0.3)

# Wykres 2: Zainteresowania (Jaccard)
axes[1].bar(
    tbl_jaccard["kategoria"], tbl_jaccard["odsetek_prawych"], color=[GRAY, PURPLE], width=0.5
)
yerr_j = [
    tbl_jaccard["odsetek_prawych"] - tbl_jaccard["ci_low_boot"],
    tbl_jaccard["ci_high_boot"] - tbl_jaccard["odsetek_prawych"],
]
axes[1].errorbar(
    tbl_jaccard["kategoria"],
    tbl_jaccard["odsetek_prawych"],
    yerr=yerr_j,
    fmt="none",
    color=GRAY,
    capsize=4,
    lw=1.5,
)
axes[1].set_title("Zainteresowania (Jaccard)")
axes[1].set_ylim(0, 0.3)

# Wykres 3: Wiek (to samo pasmo)
axes[2].bar(tbl_age["kategoria"], tbl_age["odsetek_prawych"], color=[GRAY, BLUE], width=0.5)
yerr_a = [
    tbl_age["odsetek_prawych"] - tbl_age["ci_low_boot"],
    tbl_age["ci_high_boot"] - tbl_age["odsetek_prawych"],
]
axes[2].errorbar(
    tbl_age["kategoria"],
    tbl_age["odsetek_prawych"],
    yerr=yerr_a,
    fmt="none",
    color=GRAY,
    capsize=4,
    lw=1.5,
)
axes[2].set_title("Wiek (same band)")
axes[2].set_ylim(0, 0.3)

for ax in axes:
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## 4. Główny wynik: porównanie siły predykcyjnej bloków (Styl życia vs Wiek i Zainteresowania)

**To jest sedno weryfikacji hipotezy H1.**
Zgodnie z regułą **D16** porównujemy bloki cech:
1. **Blok wieku i zainteresowań** (`same_band`, `age_band_diff`, `interests_jaccard`, `interests_shared_n`, `interests_count_diff_abs`, kontrole: `same_sex`, `same_city`).
2. **Blok stylu życia** (`lifestyle_agree_ratio`, `lifestyle_tag_conflict`, `lifestyle_shared_tags_n`, kontrole: `same_sex`, `same_city`).
3. **Model pełny** (oba bloki łącznie).

Błędy standardowe szacujemy z klastrowaniem dwustronnym po nadawcy i odbiorcy (**D18**).
Badamy przyrost dopasowania ($\Delta \text{Log-Likelihood}$, $\Delta \text{Pseudo-}R^2$) przy włączaniu bloków w **obu kolejnościach** (**D16**).


In [ ]:
h1_res = h1.fit_h1_logit_models(df)

print("=== Porównanie dopasowania bloków w modelu logistycznym ===")
display(h1_res["summary"])

d_life_ll = h1_res["delta_lifestyle_over_interests_ll"]
d_life_r2 = h1_res["r2_lifestyle_increment"]
d_int_ll = h1_res["delta_interests_over_lifestyle_ll"]
d_int_r2 = h1_res["r2_interests_increment"]

print("\n=== Oszacowanie przyrostu w obu kolejnościach dodawania (D16) ===")
print("1. Przyrost po dodaniu Stylu Życia do Wieku i Zainteresowań:")
print(f"   Delta Log-Likelihood: +{d_life_ll:.2f}")
print(f"   Delta Pseudo-R2:      +{d_life_r2:.5f}")

print("\n2. Przyrost po dodaniu Wieku i Zainteresowań do Stylu Życia:")
print(f"   Delta Log-Likelihood: +{d_int_ll:.2f}")
print(f"   Delta Pseudo-R2:      +{d_int_r2:.5f}")

ratio = d_int_ll / d_life_ll
print(
    f"\nStosunek przyrostu wiarygodności (Wiek+Zainteresowania vs Styl Życia): "
    f"{ratio:.1f}x na korzyść Wieku i Zainteresowań."
)

In [ ]:
print("=== Współczynniki i ilorazy szans (OR) w modelu pełnym (błędy klastrowane) ===")
display(
    h1_res["odds_ratios"][["zmienna", "coef", "se", "or", "ci_low", "ci_high", "p_value"]].round(4)
)

### Test na przypadek: tasowanie stylu życia odbiorców (shuffled null baseline)

**Pytanie:** Czy obserwowany przyrost dopasowania ze stylu życia ($\Delta \text{LL} \approx +41.4$) jest większy, niż mógłby wyniknąć z czystego przypadku?

**Jak sprawdzamy:** Losowo tasujemy cechy stylu życia między **odbiorcami** (całymi osobami, z zachowaniem liczby swipe'ów i popularności każdego profilu). Sprawdzamy rozkład $\Delta \text{LL}$ z 500 powtórzeń losowych (szary rozkład null) i porównujemy go z zaobserwowanym efektem rzeczywistym (kolorowa pionowa linia).


In [ ]:
obs_ll, null_lls, perm_p = h1.permutation_null_lifestyle(df, n_perm=N_PERM, seed=SEED)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(
    null_lls, bins=25, color=GRAY, alpha=0.7, edgecolor=HAIR, label="Rozkład zerowy (tasowanie)"
)
ax.axvline(
    obs_ll, color=PURPLE_INK, lw=2.5, linestyle="--", label=f"Obserwowany przyrost ({obs_ll:.1f})"
)
ax.set_xlabel("Przyrost Log-Likelihood ze stylu życia (Delta LL)")
ax.set_ylabel("Liczba permutacji")
ax.set_title(f"Test permutacji zerowej dla bloku stylu życia (p = {perm_p:.4f})")
ax.legend(frameon=True)
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

q025, q975 = np.quantile(null_lls, [0.025, 0.975])
print(f"Zaobserwowany przyrost ze stylu życia: {obs_ll:.2f}")
ci_str = f"[{q025:.2f}, {q975:.2f}]"
print(f"Średni przyrost z losowego tasowania: {null_lls.mean():.2f} (95% CI: {ci_str})")
print(f"Empiryczna wartość p: {perm_p:.4f}")

## 5. Analizy wrażliwości: czy wynik zależy od naszych decyzji?

**Pytanie:** Czy konkluzja o relacji stylu życia vs wieku i zainteresowań zmienia się w zależności od podjętych decyzji analitycznych?

Testujemy stabilność przyrostów dopasowania w 5 scenariuszach:
- **main**: próba główna (wersje as-of wg **D7**, bez Hyderabadu wg **D10**, bez odrzucania skrajnych wg **D6**),
- **tylko prior_version**: wyłącznie swipe'y ze stanem profilu odbiorcy ściśle sprzed dnia decyzji (**D7**),
- **oba prior_version**: swipe'y, gdzie obaj użytkownicy mają wersję strictly prior,
- **bez kont skrajnych**: bez użytkowników o 100% prawych lub 100% lewych swipe'ach (**D6**),
- **aktywni nadawcy**: wyłącznie nadawcy z $\ge 10$ swipe'ami (**D6**).


In [ ]:
sens_df = h1.sensitivity_analysis_h1(df)
display(sens_df.round(4))

fig, ax = plt.subplots(figsize=(9, 3.5))
x = np.arange(len(sens_df))
width = 0.35

ax.bar(
    x - width / 2,
    sens_df["delta_ll_lifestyle"],
    width,
    label="Styl życia (przyrost ponad wiek+hobby)",
    color=PURPLE,
)
ax.bar(
    x + width / 2,
    sens_df["delta_ll_interests"],
    width,
    label="Wiek i hobby (przyrost ponad styl życia)",
    color=BLUE,
)

ax.set_xticks(x)
ax.set_xticklabels(sens_df["scenariusz"], rotation=20, ha="right")
ax.set_ylabel("Delta Log-Likelihood")
ax.set_title("Analiza wrażliwości: stabilność przyrostu wiarygodności bloków")
ax.legend(frameon=True)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Czy styl życia pomaga przewidywać swipe'y u nowych osób? (Walidacja krzyżowa, D18)

**Pytanie:** Czy znając styl życia, przewidzimy prawy swipe u osób, których model **nie widział podczas uczenia**, lepiej niż znając wiek i zainteresowania?

Dzielimy nadawców na 5 niezależnych foldów (`GroupKFold` po `user_key`). Ewaluujemy na zbiorach testowych (out-of-sample) wskaźniki `logloss` oraz `AUC` dla modeli:
- `null`: model pusty,
- `age_interests`: demografia i zainteresowania,
- `lifestyle`: styl życia,
- `full`: połączone wszystkie bloki.


In [ ]:
cv_res = h1.grouped_cv_h1(df, n_splits=N_FOLDS, seed=SEED)

cv_pivot_loss = cv_res.pivot(index="fold", columns="block", values="logloss")[
    ["null", "lifestyle", "age_interests", "full"]
]
cv_pivot_auc = cv_res.pivot(index="fold", columns="block", values="auc")[
    ["null", "lifestyle", "age_interests", "full"]
]

print("--- Log Loss na zbiorach testowych (mniejszy = lepszy model) ---")
display(cv_pivot_loss)

print("\n--- AUC na zbiorach testowych (większy = lepsza separacja) ---")
display(cv_pivot_auc)

summary_cv = pd.DataFrame(
    {
        "LogLoss Mean": cv_pivot_loss.mean(),
        "LogLoss Std": cv_pivot_loss.std(),
        "AUC Mean": cv_pivot_auc.mean(),
        "AUC Std": cv_pivot_auc.std(),
    }
)
print("\n--- Średnie wyniki testowe z 5-krotnej walidacji krzyżowej ---")
display(summary_cv.round(4))

## 7. Wniosek

### Podsumowanie wyników empirycznych dla hipotezy H1

**Hipoteza H1 NIE ZOSTAŁA POTWIERDZONA (uzyskano rzetelny wynik negatywny):**
Zgodność stylu życia **nie przewiduje prawego swipe'a silniej** niż zgodność wieku i zainteresowań. W rzeczywistości obserwowany wzorzec wskazuje na dominację wieku i zainteresowań nad deklarowanym stylem życia:

| Porównywany wymiar | Samodzielny przyrost $\Delta \text{Pseudo-}R^2$ | Przyrost jako kolejny blok $\Delta \text{Pseudo-}R^2$ | Wzrost testowego AUC w CV (out-of-sample) |
|---|---|---|---|
| **Styl życia** | $+0.0040$ | $+0.0009$ | $+0.0235$ (samodzielnie) / $+0.0046$ (ponad wiek+hobby) |
| **Wiek i zainteresowania** | $+0.0187$ | $+0.0156$ | $+0.0785$ (samodzielnie) / $+0.0597$ (ponad styl życia) |

### Główne wnioski:
1. **Wiek i zainteresowania dominują**: Zgodność wieku (`same_band`, $\text{OR} = 1.19$, $p=0.031$) oraz preferencja dla osób w podobnym wieku (`age_band_diff`, $\text{OR} = 0.77$, $p < 0.001$) stanowią najsilniejszy filtr na etapie pierwszego swipe'a.
2. **Słaby sygnał ze stylu życia**: Choć blok stylu życia wykazuje statystycznie istotny przyrost wiarygodności w porównaniu z tasowaniem losowym ($p < 0.01$ w teście permutacji), jego realny wkład predykcyjny jest niemal **18-krotnie mniejszy** niż wkład wieku i zainteresowań ($\Delta \text{LL} = 41.4$ vs $739.1$).
3. **Praktyczny brak zysku uogólniania**: W 5-krotnej walidacji krzyżowej po nadawcach dodanie cech stylu życia do modelu wieku i zainteresowań obniża testowy logloss zaledwie o $0.0002$ i podnosi AUC o skromne $0.0046$.
4. **Ograniczenia metodyczne**: W aplikacji mobilnej wiek i zdjęcie profilowe są widoczne natychmiast na pierwszej karcie, podczas gdy szczegółowe pola stylu życia wymagają przewinięcia profilu. Niewielki wpływ stylu życia na etapie swipe'a nie musi oznaczać, że nie ma on znaczenia przy ostatecznym wyborze mieszkania, lecz że **na etapie selekcji w aplikacji użytkownicy filtrują przede wszystkim demografią i hobby**.
